In [0]:
%sql
CREATE OR REPLACE TABLE cms.gold.dim_state
COMMENT 'Prescriber state / area'
AS
SELECT state_code, state_name, area_type
FROM cms.silver.ref_state;

-- A dimension table describes the who, what, and where of each row. The fact table records the how much. dim_state: OH | Ohio | State

In [0]:
%sql
CREATE OR REPLACE TABLE cms.gold.dim_specialty
COMMENT 'Prescriber specialty and group'
AS
SELECT specialty, specialty_group
FROM cms.silver.ref_specialty;
CREATE OR REPLACE TABLE cms.gold.dim_drug
COMMENT 'Drug = brand name + generic name pair. drug_key is a deterministic hash of the pair.'
AS
SELECT
  xxhash64(brand_name, generic_name) AS drug_key,
  brand_name,
  generic_name
FROM (
  SELECT DISTINCT brand_name, generic_name
  FROM cms.silver.partd_prescriber_drug
);
CREATE OR REPLACE TABLE cms.gold.dim_prescriber
COMMENT 'One row per prescriber NPI'
AS
SELECT DISTINCT
  prescriber_npi,
  CASE WHEN is_organization THEN prescriber_last_or_org_name
       ELSE concat_ws(', ', prescriber_last_or_org_name, prescriber_first_name)
  END                         AS prescriber_name,
  is_organization,
  prescriber_city
FROM cms.silver.partd_prescriber_drug;

-- dim_prescriber: 1003000142 | Khalil, Rashid | Toledo
-- dim_drug: drug_key | Baclofen | Baclofen
-- dim_specialty: Anesthesiology | Other
-- In short, dimensions are what you click on to filter and slice; the fact table is what gets added up.

In [0]:
%sql
CREATE OR REPLACE TABLE cms.gold.fact_prescriber_drug
COMMENT 'Grain: one row per data_year x prescriber x drug. Beneficiaries are NULL when CMS suppressed them (1-10).'
AS
SELECT
  data_year,
  prescriber_npi,
  xxhash64(brand_name, generic_name) AS drug_key,
  state_code,
  specialty,
  total_claims,
  total_30day_fills,
  total_day_supply,
  total_drug_cost,
  total_beneficiaries,
  beneficiaries_suppressed
FROM cms.silver.partd_prescriber_drug;

-- fact: 2024 | 1003000142 | drug_key | OH | Anesthesiology | 47 | 333.35 (just the numbers and the keys)
-- The fact table can compute the key itself, without joining to dim_drug. A star schema keeps each dimension flat, exactly one join away from the fact table. 
-- 65+ columns were left out on purpose and stay in silver, because they’re too heavily hidden to be reliable.